In [1]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
import pandas as pd
from pathlib import Path

# ---------------- USER INPUTS ----------------
r_compound = r"D:\Phd Research\Final_Raster\100yr_compound_flood_base_stat.tif"
r_fema     = r"D:\Phd Research\Final_Raster\Fema_100yr_depth_200m.tif"
r_noaa     = r"C:\Users\sahad2\Research_data\SLR_Depth_Raster_NOAA\SLR_depth_3_5ft_tx_200m_UTM_m.tif"

THRESH = 0.10  # meters; pixel is inundated if depth > THRESH
# --------------------------------------------------------

def read_depth(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        nod = src.nodata
        if nod is not None:
            arr = np.where(arr == nod, np.nan, arr)
        prof = src.profile
    return arr, prof

def mask_from_depth(arr, thresh=THRESH):
    """Return 0/1 mask (uint8) where depth > thresh and finite."""
    m = (np.isfinite(arr) & (arr > thresh)).astype("uint8")
    return m

def reproject_mask_to_ref(mask_uint8, prof_src, prof_ref):
    """Nearest-neighbor reprojection of a 0/1 mask to the reference grid."""
    dst = np.zeros((prof_ref["height"], prof_ref["width"]), dtype="uint8")
    reproject(
        source=mask_uint8,
        destination=dst,
        src_transform=prof_src["transform"], src_crs=prof_src["crs"],
        dst_transform=prof_ref["transform"], dst_crs=prof_ref["crs"],
        resampling=Resampling.nearest,
    )
    return dst

def pixel_area_m2_from_profile(prof):
    # Works when prof['crs'] is projected (e.g., UTM)
    # a = pixel width, e = (negative) pixel height in transform
    a = abs(prof["transform"].a)
    e = abs(prof["transform"].e)
    return a * e

# --- Read rasters ---
D_comp, prof_comp = read_depth(r_compound)
D_fema, prof_fema = read_depth(r_fema)
D_noaa, prof_noaa = read_depth(r_noaa)

# --- Build inundation masks ---
M_comp = mask_from_depth(D_comp)                       # already on reference grid
M_fema = reproject_mask_to_ref(mask_from_depth(D_fema), prof_fema, prof_comp)
M_noaa = reproject_mask_to_ref(mask_from_depth(D_noaa), prof_noaa, prof_comp)

# --- Pixel area (km²) ---
pix_area_m2 = pixel_area_m2_from_profile(prof_comp)
pix_area_km2 = pix_area_m2 / 1e6

def compare_against_benchmark(M_bench, bench_name):
    """Return one row for the summary table comparing compound vs benchmark."""
    comp = M_comp.astype(bool)
    bench = M_bench.astype(bool)

    overlap = comp & bench
    unique_comp = comp & (~bench)
    unique_bench = bench & (~comp)
    union = comp | bench

    # Areas
    area_compound_km2 = comp.sum() * pix_area_km2
    area_overlap_km2 = overlap.sum() * pix_area_km2
    area_unique_bench_km2 = unique_bench.sum() * pix_area_km2
    area_unique_comp_km2 = unique_comp.sum() * pix_area_km2
    area_union_km2 = union.sum() * pix_area_km2

    # Percentages normalized by UNION so they sum to 100
    if area_union_km2 > 0:
        pct_overlap = 100.0 * area_overlap_km2 / area_union_km2
        pct_unique_bench = 100.0 * area_unique_bench_km2 / area_union_km2
        pct_unique_comp = 100.0 * area_unique_comp_km2 / area_union_km2
    else:
        pct_overlap = pct_unique_bench = pct_unique_comp = 0.0

    return {
        "Comparison": f"Compound vs {bench_name}",
        "Total Flooded Area (km²)": round(area_compound_km2),   # compound area
        "Overlap with Compound (%)": round(pct_overlap, 1),
        "Unique to Benchmark (%)": round(pct_unique_bench, 1),
        "Unique to Compound (%)": round(pct_unique_comp, 1),
    }

rows = []
rows.append(compare_against_benchmark(M_fema, "FEMA"))
rows.append(compare_against_benchmark(M_noaa, "NOAA SLR"))

df = pd.DataFrame(rows)

# Pretty print
with pd.option_context('display.max_columns', None, 'display.width', 120):
    print(df.to_string(index=False))

# Save CSV (optional)
out_csv = Path.cwd() / "compound_benchmark_overlap_summary.csv"
df.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")


          Comparison  Total Flooded Area (km²)  Overlap with Compound (%)  Unique to Benchmark (%)  Unique to Compound (%)
    Compound vs FEMA                     13960                       66.1                      0.8                    33.1
Compound vs NOAA SLR                     13960                       11.1                      0.0                    88.9

Saved: C:\Users\sahad2\compound_benchmark_overlap_summary.csv
